# 07 改写残差连接：value residual、超连接与记忆子层

在教程 06 中，我们读了模型的残差流。不少改模型结构的论文关注的也是这一层面的问题：比如改变每个 attention 层读到的 value（如 ResFormer 的 value residual），改变子层输出并入残差流的方式（如 LAuReL、超连接 hyper-connections），或者在子层旁边挂一个边读边写的快权重记忆（如 TTT、Titans）。

如果你想在推理服务里试验这类结构（例如拿到一个按新结构训练的检查点，或者想先验证结构接得对不对），传统做法通常是 fork 一份 vLLM 的模型代码，逐层修改 `forward` 函数；一旦涉及快权重记忆，还得自己管理每个请求独立的状态，防止并发请求之间互相串数据。

在 pllm 中，`@pllm.custom_residual` 可以让一个方法接管指定位置的残差更新，而原生的 attention、MLP 和 KV 写入照常运行。当把超参数设为恒等变换时，输出与原模型逐字相同，方便你先确认结构没有接错；对于带状态的记忆子层，每个请求拥有独立的一份记忆并按固定块长更新，数学上的结果不依赖 prompt 怎样被调度器切成步。

在开始之前需要说明两点限制与代价：
第一，本篇使用的 `Qwen/Qwen2.5-0.5B-Instruct` 并没有按这些新结构训练过。因此，把参数设为非恒等值只会让模型的输出发生变化，不会让回答质量变好。本篇演示的是如何把这些结构接进推理引擎，以及它们带来的运行开销。
第二，`@pllm.custom_residual` 需要关闭前缀缓存、在单卡上运行。它和模型一起在 CUDA graph 里运行；每层多出来的计算会让吞吐下降多少，我们在第三步实测。

本篇分四步来进行实验：
1. **value residual**：把第 0 层的 value 混入后续各层的 value。
2. **超连接（hyper-connections）**：把一条残差流拆成多条并行的流，在子层前后做读取与混合。
3. **吞吐代价**：实测这几种结构在 CUDA graph 里运行时的吞吐。
4. **带状态的记忆子层**：在 MLP 旁挂载每个请求独立的快权重矩阵，实现按块读写更新。

下面先启动一个以 eager 模式运行的原生 vLLM 作为对照组，看看它在两个测试问题上的输出。

In [1]:
import os

os.environ["VLLM_LOGGING_LEVEL"] = "ERROR"  # 只显示 vLLM 的报错，省掉启动日志

import gc
import time
from collections.abc import Mapping
from dataclasses import dataclass, replace

import torch
from transformers import AutoTokenizer
from vllm import SamplingParams

import pllm

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
CONFIG = pllm.ModelConfig(MODEL, max_model_len=512, max_num_seqs=32, max_num_batched_tokens=512,
                          gpu_memory_utilization=0.4, seed=0)

In [2]:
tokenizer = AutoTokenizer.from_pretrained(MODEL)
questions = ["Write one sentence about the ocean.", "Name three prime numbers."]
prompts = [tokenizer.apply_chat_template([{"role": "user", "content": q}], tokenize=False,
                                         add_generation_prompt=True) for q in questions]
greedy = SamplingParams(max_tokens=40, temperature=0.0)


def answers(llm) -> list[str]:
    return [output.outputs[0].text for output in llm.generate(prompts, greedy, use_tqdm=False)]


def run(workload: pllm.Workload, config: pllm.ModelConfig = CONFIG) -> list[str]:
    with pllm.make_llm(config, workload=workload) as llm:
        texts = answers(llm)
        print("eager 原因:", repr(llm.diagnostics().eager_reason))
    gc.collect()
    torch.cuda.empty_cache()
    return texts


# 对照组：不带负载的原生 vLLM，用 eager 模式运行（理由见下文）。
with pllm.make_llm(replace(CONFIG, cuda_graphs=False)) as plain:
    baseline = answers(plain)
gc.collect()
torch.cuda.empty_cache()
for text in baseline:
    print(repr(text))

'The ocean, a vast and mysterious body of water, is a vast expanse of blue and green, teeming with life and energy, where countless species of marine life thrive, from tiny plankton'
'Three prime numbers are 2, 3, and 5.'


## 第一步：value residual（改写每层的 value）

这一步我们要回答的问题是：怎样在不改模型源码的情况下，修改每一层 attention 读到并写入 KV cache 的 value，并且确认代码接法是对的？

先解释一下上面的对照组为什么特意关掉了 CUDA graph（`cuda_graphs=False`）。带 `@pllm.custom_residual` 的模型仍然在 CUDA graph 里运行，但 vLLM 不再用 inductor 融合算子，graph 里捕获的是模型原本的算子，算术和 eager 模式一致；默认的原生 vLLM 用的是融合后的算子，bf16 舍入略有不同（教程 00 讲过 eager 与 CUDA graph 的舍入差别）。所以要验证恒等设置下“与原生逐字相同”，我们和 eager 模式的原生 vLLM 对比。

我们要实现的第一个结构是 ResFormer 提出的 value residual。它的计算规则很直观：从第 1 层起，每层 attention 的 value 混入一部分第 0 层的 value，公式为 $V_l + \lambda_l (V_0 - V_l)$。当混合系数 `lam = 0` 时，公式退化为 $V_l$，也就是原模型。

在下面的代码中，我们用 `@pllm.custom_residual(site="value", layers=tuple(range(1, 24)))` 拦截第 1 到 23 层的 value：
- `site="value"` 的方法会收到当前层的 value 张量 `v`，形状为 `[行, kv 头, 头维]`，方法返回替换后的 value（当前层的 attention 会读取这个新返回值，同时它也会被写进 KV cache）。
- `pllm.value(0)` 读取同一个 token 在这次前向传播里第 0 层的 value。
- `pllm.depth()` 返回当前层号。
- 各层的混合系数 `lam` 放在一份 `InstanceState` 里，在初始化时由 `fill` 方法填入设定值。
- 传入的张量是模型的 bf16 类型；方法里先用 `.float()` 转成 float32 计算，算完再转回原类型 `.to(v.dtype)`。

In [3]:
@dataclass(frozen=True)
class ValueResidual(pllm.Workload):
    mixing: float = 0.0

    @property
    def gates(self) -> pllm.InstanceState:
        return pllm.InstanceState({"lam": lambda m: (m.layer_count,)}, initialize=self.fill)

    def fill(self, fields: Mapping[str, torch.Tensor], generator: torch.Generator) -> None:
        fields["lam"].fill_(self.mixing)

    @pllm.custom_residual(site="value", layers=tuple(range(1, 24)))
    def mix(self, v: torch.Tensor, first: torch.Tensor = pllm.value(0), layer: int = pllm.depth(),
            lam: torch.Tensor = pllm.parameter("gates", "lam")) -> torch.Tensor:
        return (v.float() + lam[layer] * (first.float() - v.float())).to(v.dtype)

In [4]:
print("lam = 0，与原生逐字相同:", run(ValueResidual(0.0)) == baseline)
for text in run(ValueResidual(0.3)):
    print("lam = 0.3:", repr(text))

eager 原因: ''


lam = 0，与原生逐字相同: True


eager 原因: ''


lam = 0.3: 'The ocean, vast and boundless, is a vast and mysterious place, teeming with life and energy, a vast expanse of blue and white, a place of endless possibilities and endless wonder.'
lam = 0.3: 'Sure, here are the first three prime numbers:\n\n1. 2\n2. 3\n3. 5'


从运行输出可以看到，每次运行的 `eager 原因` 都是空字符串：模型在 CUDA graph 里运行。当 `lam = 0` 时，输出与 eager 模式的原生 vLLM 逐字相同（`True`），这说明结构接对了。
当把 `lam` 设为 `0.3` 时，两个回答都变了：第一个回答换了说法（`'The ocean, vast and boundless, is a vast and mysterious place...'`），第二个从一句话变成了带编号的列表。回答依然通顺，但这只说明修改已经生效，不说明模型变好，因为底座模型没有按这个结构训练过。

## 第二步：超连接（把一条残差流扩成多条）

这一步我们要回答的问题是：如果一个结构不仅改子层的输出，还要把单条残差流扩展成多条并行流，并在子层前后分别做组合，该怎样用 `@pllm.custom_residual` 表达？

超连接（hyper-connections，Zhu et al., 2024）的做法是把原本的一条残差流换成 `n` 条并行的流：
1. 每个子层开始前，从 `n` 条流里按权重 `read` 加权求和读出一条，作为该子层的输入。
2. 子层算完得到输出 `y` 后，`n` 条流先按 `mix` 矩阵互相混合，再按权重 `write` 把子层输出 `y` 写回各条流。
3. 跑完所有层之后，在进入最终的 norm 之前，把 `n` 条流合成一条。

下面演示的是静态版本的超连接（权重是常数）。当取 `read = 1/n`、`mix = 单位阵`、`write = 1` 时，多条流里始终装着相同的值，行为就退化回原模型。

在代码中，我们配合使用几个参数来完成这套连接：
- `streams=pllm.Attr("n")` 让残差流的形状从 `[行, hidden]` 变成 `[行, n, hidden]`。
- `read="choose"` 指定在子层运行前调用 `choose` 方法，把 `[行, n, hidden]` 按权重压成 `[行, hidden]` 送给子层。
- `site="both"` 表示这个钩子同时作用于每一层的 attention 和 MLP 两个子层。此时 `pllm.depth()` 返回的是子层序号：attention 子层为 `2 × 层号`，MLP 子层再加 1（即 `2 × 层号 + 1`）。
- `site="final"` 挂载在所有层跑完之后、最终 norm 之前，这里的 `collapse` 方法对第 1 维求均值，把 `n` 条流重新合成一条。

顺带一提，像 LAuReL（Menghani et al., 2024）这类无需拆分成多条流、只在每个子层做残差组合的结构，同样可以用 `site="both"` 来写：直接返回 `alpha * y + beta * x + 低秩项` 即可。

In [5]:
@dataclass(frozen=True)
class HyperConnections(pllm.Workload):
    n: int = 4
    perturb: float = 0.0

    @property
    def routing(self) -> pllm.InstanceState:
        n = self.n
        return pllm.InstanceState(
            {"read": lambda m: (2 * m.layer_count, n), "mix": lambda m: (2 * m.layer_count, n, n),
             "write": lambda m: (2 * m.layer_count, n)},
            initialize=self.fill,
        )

    def fill(self, fields: Mapping[str, torch.Tensor], generator: torch.Generator) -> None:
        fields["read"].fill_(1 / self.n)
        fields["write"].fill_(1)
        eye = torch.eye(self.n, device=fields["mix"].device)
        noise = torch.randn(fields["mix"].shape, generator=torch.Generator(device=eye.device).manual_seed(5),
                            device=eye.device) * self.perturb
        fields["mix"].copy_(eye + noise - noise.mean(-1, keepdim=True))

    def choose(self, x: torch.Tensor, s: int = pllm.depth(),
               read: torch.Tensor = pllm.parameter("routing", "read")) -> torch.Tensor:
        return torch.einsum("n,rnd->rd", read[s].float(), x.float()).to(x.dtype)

    @pllm.custom_residual(site="both", read="choose", streams=pllm.Attr("n"))
    def update(self, x: torch.Tensor, y: torch.Tensor, s: int = pllm.depth(),
               mix: torch.Tensor = pllm.parameter("routing", "mix"),
               write: torch.Tensor = pllm.parameter("routing", "write")) -> torch.Tensor:
        mixed = torch.einsum("mn,rnd->rmd", mix[s].float(), x.float())
        return (mixed + write[s].float()[:, None] * y.float()[:, None]).to(x.dtype)

    @pllm.custom_residual(site="final")
    def collapse(self, x: torch.Tensor) -> torch.Tensor:
        return x.float().mean(1).to(x.dtype)

In [6]:
print("恒等路由，与原生逐字相同:", run(HyperConnections()) == baseline)
for text in run(HyperConnections(perturb=0.5)):
    print("扰动 0.5:", repr(text))

eager 原因: ''


恒等路由，与原生逐字相同: True


eager 原因: ''


扰动 0.5: 'The ocean, vast and deep, is a vast and deep body of water that surrounds our planet, teeming with life and providing vital resources to countless civilizations and cultures around the world.'
扰动 0.5: 'Sure, here are three prime numbers:\n\n1. 2\n2. 3\n3. 5\n\nThese are the smallest prime numbers, and they are all prime numbers themselves. Prime numbers'


来看运行结果：在恒等路由（`perturb=0.0`，即 `read = 1/n`、`mix = 单位阵`、`write = 1`）下，4 条并行流算出的结果依然与 eager 原生模型逐字相同（`True`）。当给混合矩阵加上 `0.5` 的随机扰动后，第一个回答换了说法（变成了 `'The ocean, vast and deep, is a vast and deep body of water...'`），第二个变成了带编号的列表，并多了一段解释；两者仍然通顺。

## 第三步：实测吞吐代价

这一步我们要回答的问题是：这些结构在 CUDA graph 里运行时，每层新加的张量计算会让推理吞吐降低多少？

为了测出真实吞吐，我们构造 32 个并发请求（8 个 prompt × 每个采样 4 个样本 × 生成 256 个 token），先预热一轮，再同步计时一轮。我们把四组配置放在一起对比：开启 CUDA graph 的原生 vLLM、eager 模式的原生 vLLM（作为参照，看看如果失去 CUDA graph 会是多少）、刚才写的 `ValueResidual(0.3)`，以及 4 条流的 `HyperConnections(perturb=0.5)`。

In [7]:
many = [f"Explain the mathematical idea numbered {i}:" for i in range(8)]
sampling = SamplingParams(n=4, max_tokens=256, temperature=0.8, top_p=0.95, ignore_eos=True, seed=7)


def throughput(llm) -> float:
    llm.generate(many, sampling, use_tqdm=False)  # 预热
    torch.cuda.synchronize()
    start = time.perf_counter()
    outputs = llm.generate(many, sampling, use_tqdm=False)
    torch.cuda.synchronize()
    return sum(len(sample.token_ids) for output in outputs for sample in output.outputs) / (time.perf_counter() - start)


gc.collect()  # 前面几个引擎关闭后，先把它们占的显存还给 GPU
torch.cuda.empty_cache()
rates = {}
for name, config in (("原生 vLLM，CUDA graph", CONFIG), ("原生 vLLM，eager", replace(CONFIG, cuda_graphs=False))):
    with pllm.make_llm(config) as plain:
        rates[name] = throughput(plain)
    gc.collect()
    torch.cuda.empty_cache()
for name, workload in (("value residual", ValueResidual(0.3)), ("超连接 n=4", HyperConnections(perturb=0.5))):
    with pllm.make_llm(CONFIG, workload=workload) as llm:
        rates[name] = throughput(llm)
    gc.collect()
    torch.cuda.empty_cache()
graph = rates["原生 vLLM，CUDA graph"]
for name, rate in rates.items():
    print(f"{name:22s}: {rate:6.0f} token/s，为 CUDA graph 原生的 {rate / graph:.0%}")

原生 vLLM，CUDA graph    :  13267 token/s，为 CUDA graph 原生的 100%
原生 vLLM，eager         :   6022 token/s，为 CUDA graph 原生的 45%
value residual        :  11305 token/s，为 CUDA graph 原生的 85%
超连接 n=4               :   8570 token/s，为 CUDA graph 原生的 65%


看一下实测数据（基于 0.5B 模型、32 个并发请求）：
- **原生 vLLM，CUDA graph**：`13267 token/s`（作为 `100%` 基准）
- **原生 vLLM，eager**：`6022 token/s`，为 CUDA graph 原生的 `45%`
- **value residual**：`11305 token/s`，为 CUDA graph 原生的 `85%`
- **超连接 n=4**：`8570 token/s`，为 CUDA graph 原生的 `65%`

两个负载都远高于 eager 原生的 `45%`，说明它们确实留在了 CUDA graph 里。剩下的差距来自每层多出来的计算：value residual 修改 23 层的 value 张量，吞吐为 `85%`；超连接在 48 个子层（24 层 × 每层 2 个子层）上各自做 4 条残差流的读取与矩阵混合，吞吐为 `65%`。

## 第四步：带状态的记忆子层（按块更新的快权重）

这一步我们要回答的问题是：如果残差分支里带有需要随 token 边读边写的“记忆”（每个请求独立一份），怎样写才能既让并发请求互不干扰，又保证计算结果不随调度器的分步方式而改变？

我们先用白话描述一下这个记忆子层在算什么：在第 12 层 MLP 旁边放一个形状为 `[hidden, hidden]` 的快权重矩阵 $M$（每个请求各自拥有一份，初始为零）。
- **读取**：每个经过第 12 层的 token，先把自己的残差流输入 $x$ 做归一化得到键 $k$，从矩阵里读出 $M k$，乘上一个门控系数 `gate`（演示中设为 `2`），加到 MLP 输出后的残差流里。
- **写入（更新）**：每凑满固定长度的一块（这里设 `chunk=16`，即 16 个 token），矩阵 $M$ 就对着目标 $\|M k - y\|^2$（其中 $y$ 是这一层 MLP 的输出）走一步梯度下降（学习率设为 `1.0`）。

这就是 TTT（Test-Time Training）或 Titans 一类“边读边写的快权重记忆”的骨架。当 `gate = 0` 时，读出的记忆项被乘零丢弃，模型退化为原模型。

在代码中，我们通过两个关键参数把这个规则交给框架管理：
- `state="memory"` 指定关联一份只有一个字段 `M` 的 `RequestState`，框架会为每个请求独立分配并隔离这块状态。
- `chunk=pllm.Attr("chunk")` 指定块长为 16。被装饰的 `remember` 方法返回元组 `(新的残差流, 新的 M)`。

关于按块更新的时机，有一条明确的规则：同一个块（16 行）里的每一行，在计算读取时用的都是**这一块开始时的 $M$**；只有当这一块的最后一行跑完时，方法返回的**新的 $M$** 才会生效并留给下一块使用。如果在某次引擎前向步中，某个请求剩下的 token 数没凑满 16 行，这些没凑满的行会由框架自动保存起来，等到后续步凑满一整块时再触发状态更新。因此从数学上讲，最终生成的结果不依赖 vLLM 调度器怎样把一段长 prompt 切成多个步来执行。

In [8]:
@dataclass(frozen=True)
class FastWeightMemory(pllm.Workload):
    layer: int = 12
    chunk: int = 16
    lr: float = 1.0
    gate: float = 2.0

    @property
    def memory(self) -> pllm.RequestState:
        return pllm.RequestState({"M": lambda m: (m.hidden_size, m.hidden_size)})

    @property
    def where(self) -> tuple[int, ...]:
        return (self.layer,)

    @pllm.custom_residual(site="mlp", layers=pllm.Attr("where"), state="memory", chunk=pllm.Attr("chunk"))
    def remember(self, x: torch.Tensor, y: torch.Tensor,
                 M: torch.Tensor = pllm.parameter("memory", "M")) -> tuple[torch.Tensor, torch.Tensor]:
        k = torch.nn.functional.normalize(x.float(), dim=-1)  # 这一行的键
        read = k @ M.T                                         # 从记忆里读出的值
        stream = (x.float() + y.float() + self.gate * read).to(x.dtype)
        return stream, M - self.lr / len(x) * (read - y.float()).T @ k  # 对 |M k - y|^2 走一步梯度

In [9]:
# 带状态的记忆子层也在 CUDA graph 里运行，需要更大的捕获预算。
MEMORY_CONFIG = replace(CONFIG, execution=pllm.ExecutionOptions(cuda_graph_pool_budget_bytes=1 << 30))
print("gate = 0，与原生逐字相同:", run(FastWeightMemory(gate=0.0), MEMORY_CONFIG) == baseline)
for text in run(FastWeightMemory(), MEMORY_CONFIG):
    print("gate = 2:", repr(text))
with pllm.make_llm(MEMORY_CONFIG, workload=FastWeightMemory()) as llm:
    rate = throughput(llm)
print(f"记忆子层: {rate:6.0f} token/s，为 CUDA graph 原生的 {rate / graph:.0%}")

eager 原因: ''


gate = 0，与原生逐字相同: True


eager 原因: ''


gate = 2: "The ocean is a vast and mysterious place, a vast expanse of water that covers the Earth's surface, teeming with life and energy."
gate = 2: 'Three prime numbers are 2, 3, and 5.'


记忆子层:   5378 token/s，为 CUDA graph 原生的 41%


从输出可以看到，当 `gate = 0` 时，输出依然与 eager 模式的原生模型逐字相同（`True`）。当把 `gate` 设为 `2` 时，第一个回答换成了 `"The ocean is a vast and mysterious place..."`，第二个回答保持不变。带状态的记忆子层同样在 CUDA graph 里运行（`eager 原因` 为空），它需要更大的捕获预算（这里设了 1 GiB，不够时 pllm 会报错并给出需要的字节数）；同样 32 个并发请求下吞吐为 `5378 token/s`，为 CUDA graph 原生的 `41%`（这个子层每个 token 读一次 `[hidden, hidden]` 的矩阵，每 16 个 token 更新一次）。

## 小结

回顾本篇的四个实验，关于 `@pllm.custom_residual` 的用法与边界可以总结为以下几点：
1. **五种挂载位置（`site`）**：可以挂载在 `attention`、`mlp`、`both`（同时作用于每层的 attention 和 MLP）、`value`（替换 attention 读取并写入 KV cache 的 value）以及 `final`（所有层之后、最终 norm 之前，常用于把多条流合并回一条）。
2. **先做恒等校验**：无论是混入首层 value、拆分多条并行流，还是外挂快权重记忆，都可以先通过参数设置成恒等变换，与 eager 模式的原生模型比对是否逐字相同，以此确认结构接法无误。
3. **带状态的记忆子层**：通过绑定 `state`（一份单字段的 `RequestState`）和 `chunk`（块长），可以实现每个请求一份状态、按固定块长读写更新的快权重结构。
4. **当前限制与开销**：`@pllm.custom_residual` 和模型一起在 CUDA graph 里运行，代价是每层多出来的计算（本篇实测 85%、65% 和 41%）；它需要单卡并关闭前缀缓存；此外，带状态（`state`）的记忆子层不能和 `@pllm.custom_forward` 一起使用，因为在自定义前向传播中，每个 token 访问同一层的次数是不固定的。

本篇是在普通模型上演示快权重记忆的骨架。在下一篇 [08 In-Place TTT](08_inplace_ttt.ipynb) 中，我们将加载一个真正按“按块更新快权重”训练出来的模型——In-Place TTT，看看它在真实权重下的运行表现。